# 🎙️ Speech Emotion Recognition — Colab Test Notebook

This notebook mirrors every functionality in `emotion_app.py`:

| Section | What it tests |
|---|---|
| **1. Install & Imports** | All dependencies |
| **2. Audio Helpers** | `to_wav_bytes`, `load_and_clean` |
| **3. Feature Extraction** | `extract_features_classical` |
| **4. Model Definitions** | `Wav2Vec2ForEmotionClassification` architecture |
| **5. Model Loading** | Load MLP + wav2vec weights from Drive / local |
| **6. Prediction** | `predict_classical`, `predict_wav2vec` |
| **7. Upload & Analyse** | File-upload widget → full pipeline |
| **8. Microphone Record** | In-browser recording → prediction |
| **9. Batch / Stress Test** | Run multiple files, measure latency |

> **Tip:** Run cells top-to-bottom. Sections 5-9 require your trained model files.


## 1 · Install Dependencies

In [1]:
# Install all packages used in emotion_app.py
!pip install -q librosa soundfile noisereduce joblib transformers pydub                torch torchaudio ipywidgets

# ffmpeg is required by pydub for non-WAV formats
!apt-get install -qq -y ffmpeg > /dev/null 2>&1
print("✅ All packages installed")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 53.7 MB/s eta 0:00:00
✅ All packages installed


In [2]:
import os, io, json, warnings, tempfile, time
warnings.filterwarnings("ignore")

import numpy as np
import soundfile as sf
import librosa
import noisereduce as nr
import joblib
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from pydub import AudioSegment

# ── same constants as emotion_app.py ──────────────────
SR_CLASSICAL = 22050
SR_WAV2VEC   = 16000
MAX_DURATION = 4.0
PRE_EMPHASIS = 0.97
TRIM_TOP_DB  = 30

EMOTION_EMOJI = {
    "angry":   "😡", "disgust": "🤢", "fear":    "😨",
    "happy":   "😄", "neutral": "😐", "ps":       "😲",
    "sad":     "😢", "surprise":"😲",
}

EMOTION_COLORS = {
    "angry":"#FF4444","disgust":"#8B4513","fear":"#800080",
    "happy":"#FFD700","neutral":"#607D8B","ps":"#FF6B35",
    "sad":"#5B86E5","surprise":"#FF9800",
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Imports OK  |  device = {device}")


✅ Imports OK  |  device = cpu


## 3 · Audio Helper Functions

In [3]:
# ── Exact copies from emotion_app.py ──────────────────

def to_wav_bytes(file_like, filename: str) -> bytes:
    """Convert any audio format to WAV bytes using pydub."""
    ext = os.path.splitext(filename)[-1].lower().lstrip(".")
    audio = AudioSegment.from_file(io.BytesIO(file_like.read()), format=ext or "wav")
    buf = io.BytesIO()
    audio.export(buf, format="wav")
    return buf.getvalue()


def load_and_clean(audio_bytes: bytes, target_sr: int) -> np.ndarray:
    buf = io.BytesIO(audio_bytes)
    y, sr = librosa.load(buf, sr=target_sr, mono=True)
    noise_clip = y[:int(0.5 * sr)] if len(y) > int(0.5 * sr) else y
    y = nr.reduce_noise(y=y, sr=sr, y_noise=noise_clip,
                        prop_decrease=0.75, stationary=False)
    y, _ = librosa.effects.trim(y, top_db=TRIM_TOP_DB)
    if len(y) == 0:
        buf.seek(0)
        y, sr = librosa.load(buf, sr=target_sr, mono=True)
    y = np.append(y[0], y[1:] - PRE_EMPHASIS * y[:-1])
    rms = np.sqrt(np.mean(y ** 2))
    if rms > 0:
        y = y * (0.1 / rms)
    return y.astype(np.float32)

print("✅ Audio helpers defined")


✅ Audio helpers defined


### 3a · Smoke-test: synthetic 1-second sine wave

In [4]:
# Generate a 440 Hz sine, save to WAV bytes, then run load_and_clean
t = np.linspace(0, 1, SR_CLASSICAL, dtype=np.float32)
sine = (0.3 * np.sin(2 * np.pi * 440 * t)).astype(np.float32)

buf = io.BytesIO()
sf.write(buf, sine, SR_CLASSICAL, format="WAV")
test_bytes = buf.getvalue()

y_clean = load_and_clean(test_bytes, SR_CLASSICAL)
print(f"Input  samples : {len(sine)}")
print(f"Output samples : {len(y_clean)}  (trimmed + pre-emphasis)")
print(f"RMS after norm : {np.sqrt(np.mean(y_clean**2)):.4f}  (target ≈ 0.1)")


Input  samples : 22050
Output samples : 22050  (trimmed + pre-emphasis)
RMS after norm : 0.1000  (target ≈ 0.1)


## 4 · Feature Extraction

In [5]:
def extract_features_classical(y: np.ndarray, sr: int) -> np.ndarray:
    feats = []
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    feats.append(np.mean(mfccs, axis=1))
    feats.append(np.mean(librosa.feature.delta(mfccs), axis=1))
    stft_mag = np.abs(librosa.stft(y))
    feats.append(np.mean(librosa.feature.chroma_stft(S=stft_mag, sr=sr), axis=1))
    feats.append(np.mean(librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128), axis=1))
    feats.append([np.mean(librosa.feature.zero_crossing_rate(y))])
    feats.append([np.mean(librosa.feature.rms(y=y))])
    return np.hstack(feats).reshape(1, -1)

# Quick test
feats = extract_features_classical(y_clean, SR_CLASSICAL)
print(f"Feature vector shape : {feats.shape}  "
      f"(40 MFCCs + 40 delta + 12 chroma + 128 mel + 1 ZCR + 1 RMS = {feats.shape[1]})")


Feature vector shape : (1, 222)  (40 MFCCs + 40 delta + 12 chroma + 128 mel + 1 ZCR + 1 RMS = 222)


## 5 · wav2vec 2.0 Model Architecture

In [6]:
class Wav2Vec2ForEmotionClassification(nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.3):
        super().__init__()
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(model_name)
        self.wav2vec2.feature_extractor._freeze_parameters()
        hidden_size = self.wav2vec2.config.hidden_size
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size), nn.Dropout(dropout),
            nn.Linear(hidden_size, 256), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(256, num_classes)
        )

    def forward(self, input_values, attention_mask=None):
        out    = self.wav2vec2(input_values=input_values, attention_mask=attention_mask)
        pooled = out.last_hidden_state.mean(dim=1)
        return self.classifier(pooled)

print("✅ Wav2Vec2ForEmotionClassification defined")


✅ Wav2Vec2ForEmotionClassification defined


## 6 · Load Trained Models

Upload your model files **or** mount Google Drive first.

Expected files (same defaults as the Streamlit app):
```
ser_mlp_model.pkl
ser_scaler.pkl
ser_label_encoder.pkl
best_wav2vec2_ser.pt
wav2vec2_config.json
wav2vec2_label_encoder.pkl
```


In [7]:
# ── Option A: mount Google Drive ──────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# MODEL_DIR = "/content/drive/MyDrive/ser_models"   # ← change to your folder

# ── Option B: upload directly ─────────────────────────
from google.colab import files as colab_files
print("Upload your 6 model files (pkl / pt / json) …")
uploaded_models = colab_files.upload()
MODEL_DIR = "/content"
print("Uploaded:", list(uploaded_models.keys()))


Upload your 6 model files (pkl / pt / json) …


Uploaded: []


In [8]:
# ── Paths ──────────────────────────────────────────────
mlp_path    = os.path.join(MODEL_DIR, "ser_mlp_model.pkl")
scaler_path = os.path.join(MODEL_DIR, "ser_scaler.pkl")
le_mlp_path = os.path.join(MODEL_DIR, "ser_label_encoder.pkl")
w2v_weights = os.path.join(MODEL_DIR, "best_wav2vec2_ser.pt")
w2v_config  = os.path.join(MODEL_DIR, "wav2vec2_config.json")
le_w2v_path = os.path.join(MODEL_DIR, "wav2vec2_label_encoder.pkl")

# Verify all present
missing = [p for p in [mlp_path, scaler_path, le_mlp_path,
                        w2v_weights, w2v_config, le_w2v_path]
           if not os.path.exists(p)]
if missing:
    print("❌ Missing files:")
    for m in missing: print(" •", m)
else:
    print("✅ All model files found")


✅ All model files found


In [9]:
# ── Load models ────────────────────────────────────────
mlp    = joblib.load(mlp_path)
scaler = joblib.load(scaler_path)
le_mlp = joblib.load(le_mlp_path)

with open(w2v_config) as f:
    w2v_cfg = json.load(f)

le_w2v    = joblib.load(le_w2v_path)
processor = Wav2Vec2Processor.from_pretrained(w2v_cfg["model_name"])
w2v_model = Wav2Vec2ForEmotionClassification(
    w2v_cfg["model_name"], w2v_cfg["num_classes"]
).to(device)
w2v_model.load_state_dict(torch.load(w2v_weights, map_location=device))
w2v_model.eval()

print(f"✅ MLP loaded        — classes: {list(le_mlp.classes_)}")
print(f"✅ wav2vec loaded    — classes: {list(le_w2v.classes_)}")
print(f"   wav2vec backbone : {w2v_cfg['model_name']}")


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

✅ MLP loaded        — classes: [np.str_('angry'), np.str_('disgust'), np.str_('fear'), np.str_('happy'), np.str_('neutral'), np.str_('ps'), np.str_('sad')]
✅ wav2vec loaded    — classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'ps', 'sad']
   wav2vec backbone : facebook/wav2vec2-base


## 7 · Prediction Functions

In [10]:
def predict_classical(audio_bytes: bytes):
    y = load_and_clean(audio_bytes, SR_CLASSICAL)
    feats = extract_features_classical(y, SR_CLASSICAL)
    feats_scaled = scaler.transform(feats)
    proba = mlp.predict_proba(feats_scaled)[0]
    pred_label = le_mlp.classes_[np.argmax(proba)]
    probs = {cls: float(p) for cls, p in zip(le_mlp.classes_, proba)}
    return pred_label, probs


def predict_wav2vec(audio_bytes: bytes):
    max_len   = w2v_cfg["max_length"]
    target_sr = w2v_cfg["target_sr"]
    y = load_and_clean(audio_bytes, target_sr)
    if len(y) < max_len:
        y = np.pad(y, (0, max_len - len(y)), mode="constant")
    else:
        y = y[:max_len]
    inputs = processor(y, sampling_rate=target_sr, return_tensors="pt", padding=False)
    input_values = inputs["input_values"].to(device)
    with torch.no_grad():
        proba = F.softmax(w2v_model(input_values), dim=-1).cpu().numpy()[0]
    pred_label = le_w2v.classes_[np.argmax(proba)]
    probs = {cls: float(p) for cls, p in zip(le_w2v.classes_, proba)}
    return pred_label, probs


def print_results(label, probs, model_name="Model"):
    emoji = EMOTION_EMOJI.get(label, "🎭")
    print(f"\n{'═'*40}")
    print(f"  {model_name}")
    print(f"  Prediction : {emoji} {label.upper()}  ({probs[label]:.1%})")
    print(f"{'─'*40}")
    for e, p in sorted(probs.items(), key=lambda x: -x[1]):
        bar = "█" * int(p * 30)
        print(f"  {EMOTION_EMOJI.get(e,'  ')} {e:<10} {bar:<30} {p:.1%}")
    print(f"{'═'*40}")

print("✅ Prediction functions defined")


✅ Prediction functions defined


## 8 · Upload Audio File → Full Pipeline

Supports: `wav`, `mp3`, `ogg`, `flac`, `m4a`, `aac`, `wma`, `aiff`


In [16]:
from google.colab import files as colab_files
from IPython.display import Audio, display

print("Upload an audio file to analyse …")
up = colab_files.upload()

for fname, fbytes in up.items():
    print(f"\n🎵 File : {fname}")

    # convert to WAV (replicates to_wav_bytes)
    ext = os.path.splitext(fname)[-1].lower().lstrip(".")
    audio_seg = AudioSegment.from_file(io.BytesIO(fbytes), format=ext or "wav")
    wav_buf = io.BytesIO()
    audio_seg.export(wav_buf, format="wav")
    wav_bytes = wav_buf.getvalue()
    print(f"   WAV size : {len(wav_bytes)//1024} KB")

    # play in notebook
    display(Audio(wav_bytes, rate=SR_CLASSICAL))

    # ── Classical MLP
    t0 = time.time()
    label_mlp, probs_mlp = predict_classical(wav_bytes)
    print_results(label_mlp, probs_mlp, f"Classical MLP  ({time.time()-t0:.2f}s)")

    # ── wav2vec 2.0
    t0 = time.time()
    label_w2v, probs_w2v = predict_wav2vec(wav_bytes)
    print_results(label_w2v, probs_w2v, f"wav2vec 2.0    ({time.time()-t0:.2f}s)")


Upload an audio file to analyse …


Saving Enregistrement (5).m4a to Enregistrement (5).m4a

🎵 File : Enregistrement (5).m4a
   WAV size : 776 KB



════════════════════════════════════════
  Classical MLP  (0.16s)
  Prediction : 🤢 DISGUST  (100.0%)
────────────────────────────────────────
  🤢 disgust    █████████████████████████████  100.0%
  😲 ps                                        0.0%
  😄 happy                                     0.0%
  😨 fear                                      0.0%
  😡 angry                                     0.0%
  😐 neutral                                   0.0%
  😢 sad                                       0.0%
════════════════════════════════════════

════════════════════════════════════════
  wav2vec 2.0    (1.17s)
  Prediction : 😨 FEAR  (61.1%)
────────────────────────────────────────
  😨 fear       ██████████████████             61.1%
  😲 ps         ██████                         22.3%
  😄 happy      █                              6.3%
  😡 angry                                     3.2%
  🤢 disgust                                   3.0%
  😐 neutral                                   2.1%
  😢 sad   

## 9 · Record from Microphone (browser)

Click **▶ Run cell**, then click the **Record** button that appears.  
When done, click **Stop** and wait for the prediction.


In [12]:
from IPython.display import Javascript, display, Audio
from google.colab.output import eval_js
import base64

RECORD_JS = """
async function recordAudio(seconds) {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  const recorder = new MediaRecorder(stream);
  const chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();

  await new Promise(resolve => setTimeout(resolve, seconds * 1000));
  recorder.stop();
  await new Promise(resolve => recorder.onstop = resolve);
  stream.getTracks().forEach(t => t.stop());

  const blob = new Blob(chunks, { type: 'audio/webm' });
  const reader = new FileReader();
  reader.readAsDataURL(blob);
  await new Promise(resolve => reader.onloadend = resolve);
  return reader.result.split(',')[1];
}

const b64 = await recordAudio(DURATION);
b64;
"""

RECORD_SECONDS = 4   # ← change recording length here

print(f"🎤 Recording {RECORD_SECONDS}s … (browser will ask for mic permission)")
b64_audio = eval_js(RECORD_JS.replace("DURATION", str(RECORD_SECONDS)))
mic_bytes  = base64.b64decode(b64_audio)

# pydub decodes webm → WAV
audio_seg = AudioSegment.from_file(io.BytesIO(mic_bytes))
wav_buf   = io.BytesIO()
audio_seg.export(wav_buf, format="wav")
wav_bytes = wav_buf.getvalue()

print(f"✅ Recorded  |  WAV size: {len(wav_bytes)//1024} KB")
display(Audio(wav_bytes, rate=SR_CLASSICAL))

label_mlp, probs_mlp = predict_classical(wav_bytes)
print_results(label_mlp, probs_mlp, "Classical MLP")

label_w2v, probs_w2v = predict_wav2vec(wav_bytes)
print_results(label_w2v, probs_w2v, "wav2vec 2.0")


🎤 Recording 4s … (browser will ask for mic permission)


MessageError: SyntaxError: await is only valid in async functions and the top level bodies of modules

## 10 · Batch / Latency Stress Test

Provide a folder path or a list of audio file paths to run all of them
through both models and get a timing summary.


In [ ]:
import glob, pandas as pd

# ── set this to a folder with audio files, or list paths explicitly
AUDIO_GLOB = "/content/*.wav"   # e.g. "/content/drive/MyDrive/test_audio/*.wav"

files = sorted(glob.glob(AUDIO_GLOB))
if not files:
    print("No files matched — upload some .wav files or change AUDIO_GLOB")
else:
    rows = []
    for path in files:
        with open(path, "rb") as f:
            audio_bytes = f.read()

        t0 = time.time(); lm, pm = predict_classical(audio_bytes); tm = time.time()-t0
        t0 = time.time(); lw, pw = predict_wav2vec(audio_bytes);   tw = time.time()-t0

        rows.append({
            "file":         os.path.basename(path),
            "MLP pred":     lm,
            "MLP conf":     f"{pm[lm]:.1%}",
            "MLP ms":       f"{tm*1000:.0f}",
            "w2v pred":     lw,
            "w2v conf":     f"{pw[lw]:.1%}",
            "w2v ms":       f"{tw*1000:.0f}",
            "agree":        "✅" if lm == lw else "❌",
        })
        print(f"  {os.path.basename(path):<30}  MLP={lm:<9}  w2v={lw:<9}  {'✅' if lm==lw else '❌'}")

    df = pd.DataFrame(rows)
    print("\n", df.to_string(index=False))
    print(f"\nAgreement rate : {(df['agree']=='✅').mean():.1%}")


## 11 · Unit Tests

In [ ]:
import unittest

class TestAudioHelpers(unittest.TestCase):

    def _sine_bytes(self, freq=440, duration=1.0, sr=SR_CLASSICAL):
        t = np.linspace(0, duration, int(sr * duration), dtype=np.float32)
        y = (0.3 * np.sin(2 * np.pi * freq * t)).astype(np.float32)
        buf = io.BytesIO()
        sf.write(buf, y, sr, format="WAV")
        return buf.getvalue()

    def test_load_and_clean_output_type(self):
        y = load_and_clean(self._sine_bytes(), SR_CLASSICAL)
        self.assertIsInstance(y, np.ndarray)
        self.assertEqual(y.dtype, np.float32)

    def test_load_and_clean_rms_normalisation(self):
        y = load_and_clean(self._sine_bytes(), SR_CLASSICAL)
        rms = np.sqrt(np.mean(y ** 2))
        # RMS should be close to 0.1 (tolerance for very short clips)
        self.assertAlmostEqual(rms, 0.1, delta=0.05)

    def test_feature_shape(self):
        y = load_and_clean(self._sine_bytes(), SR_CLASSICAL)
        feats = extract_features_classical(y, SR_CLASSICAL)
        # 40 + 40 + 12 + 128 + 1 + 1 = 222
        self.assertEqual(feats.shape[1], 222)

    def test_feature_no_nan(self):
        y = load_and_clean(self._sine_bytes(), SR_CLASSICAL)
        feats = extract_features_classical(y, SR_CLASSICAL)
        self.assertFalse(np.any(np.isnan(feats)))

    def test_wav2vec_architecture(self):
        # instantiate with a tiny config — no GPU needed
        model = Wav2Vec2ForEmotionClassification("facebook/wav2vec2-base", num_classes=8)
        dummy = torch.zeros(1, SR_WAV2VEC)           # 1 second of silence
        logits = model(dummy)
        self.assertEqual(logits.shape, (1, 8))

suite = unittest.TestLoader().loadTestsFromTestCase(TestAudioHelpers)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
